# Decision Tree tuning

This notebook uses an interpretable road/vehicle domain contract and one-hot encoding so the
tree does not invent an order between category codes. It evaluates the complete explicit grid
for depth, leaf size, split size, pruning, feature subsampling, criterion, and class weights.
The selection metric is grouped macro-F1. scikit-learn Decision Trees are CPU based; the
Windows GPU is not used.

In [2]:
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, recall_score, make_scorer
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
import sys
sys.path.insert(0, str(Path.cwd() / 'hyper parameter')); sys.path.insert(0, str(Path.cwd()))
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd()
if not (ROOT / 'dataset').exists():
    ROOT = ROOT.parent
TRAIN_PATH = ROOT / 'dataset' / 'processed' / 'merged_train.csv'
TEST_PATH = ROOT / 'dataset' / 'processed' / 'merged_test.csv'
RESULTS = ROOT / 'hyper parameter' / 'results' / 'decision_tree'
BEST = ROOT / 'hyper parameter' / 'best'
MODEL_DIR = ROOT / 'models' / 'saved_models'
RESULTS.mkdir(parents=True, exist_ok=True)
BEST.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SEED = 19
TARGET = 'casualty_severity'
GROUP = 'collision_index'
MODEL_NAME = 'decision_tree'

In [3]:
FEATURES = [
    'casualty_class', 'sex_of_casualty', 'age_of_casualty', 'casualty_type',
    'number_of_vehicles', 'day_of_week', 'first_road_class', 'road_type',
    'speed_limit', 'junction_detail', 'junction_control', 'light_conditions',
    'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area',
    'vehicle_type', 'vehicle_manoeuvre', 'vehicle_direction_from',
    'vehicle_direction_to', 'vehicle_leaving_carriageway',
    'first_point_of_impact', 'hit_object_off_carriageway', 'age_of_driver',
    'engine_capacity_cc', 'journey_purpose_of_driver',
]
NUMERIC = ['age_of_casualty', 'age_of_driver', 'engine_capacity_cc', 'number_of_vehicles', 'speed_limit']
CATEGORICAL = [column for column in FEATURES if column not in NUMERIC]
train = pd.read_csv(TRAIN_PATH, dtype={GROUP: str})
test = pd.read_csv(TEST_PATH, dtype={GROUP: str})
missing = sorted(set(FEATURES) - set(train.columns))
if missing:
    raise ValueError(f'Missing prepared columns: {missing}. Run build_dataset.py first.')
X = train[FEATURES]
y = train[TARGET].astype(int)
groups = train[GROUP].astype(str)
X_test = test[FEATURES]
y_test = test[TARGET].astype(int)
print('train:', X.shape, 'test:', X_test.shape)
print('class proportions:', y.value_counts(normalize=True).sort_index().round(6).to_dict())

train: (102305, 25) test: (25578, 25)
class proportions: {1: 0.012023, 2: 0.217057, 3: 0.77092}


In [4]:
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', min_frequency=10), CATEGORICAL),
    ('numeric', 'passthrough', NUMERIC),
], remainder='drop', sparse_threshold=1.0)
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=SEED)),
])
PARAM_GRID = {
    'classifier__max_depth': [4, 6, 8, 12, 16, 24, None],
    'classifier__min_samples_leaf': [1, 2, 5, 10, 20, 50],
    'classifier__min_samples_split': [2, 5, 10, 25, 50],
    'classifier__max_features': [None, 'sqrt', 0.7],
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__ccp_alpha': [0.0, 0.00001, 0.0001, 0.001],
    'classifier__class_weight': [None, 'balanced', {1: 3, 2: 1.5, 3: 1}],
}
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_splits = list(cv.split(X, y, groups))
for fold, (_, validation_index) in enumerate(cv_splits):
    fold_rate = y.iloc[validation_index].eq(1).mean()
    if abs(fold_rate - y.eq(1).mean()) > 0.001:
        raise ValueError(f'Fatal proportion is outside tolerance in fold {fold}')
print('folds:', len(cv_splits), 'parameter combinations:', 7 * 6 * 5 * 3 * 2 * 4 * 3)

folds: 5 parameter combinations: 15120


In [6]:
scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'macro_f1': 'f1_macro',
    'fatal_recall': make_scorer(recall_score, labels=[1], average='macro', zero_division=0),
    'serious_recall': make_scorer(recall_score, labels=[2], average='macro', zero_division=0),
    'slight_recall': make_scorer(recall_score, labels=[3], average='macro', zero_division=0),
}
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=PARAM_GRID,
    scoring=scoring,
    refit='macro_f1', n_iter=24, random_state=SEED,
    cv=cv_splits,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
    error_score='raise',
)
search.fit(X, y, groups=groups)
cv_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_macro_f1')
cv_results.to_csv(RESULTS / 'cv_results.csv', index=False)
print('best parameters:', search.best_params_)
print('best grouped macro-F1:', search.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV] END classifier__ccp_alpha=0.001, classifier__class_weight={1: 3, 2: 1.5, 3: 1}, classifier__criterion=gini, classifier__max_depth=4, classifier__max_features=sqrt, classifier__min_samples_leaf=50, classifier__min_samples_split=10; total time=   0.4s
[CV] END classifier__ccp_alpha=0.001, classifier__class_weight={1: 3, 2: 1.5, 3: 1}, classifier__criterion=gini, classifier__max_depth=4, classifier__max_features=sqrt, classifier__min_samples_leaf=50, classifier__min_samples_split=10; total time=   0.1s
[CV] END classifier__ccp_alpha=0.001, classifier__class_weight={1: 3, 2: 1.5, 3: 1}, classifier__criterion=gini, classifier__max_depth=4, classifier__max_features=sqrt, classifier__min_samples_leaf=50, classifier__min_samples_split=10; total time=   0.2s
[CV] END classifier__ccp_alpha=0.001, classifier__class_weight={1: 3, 2: 1.5, 3: 1}, classifier__criterion=gini, classifier__max_depth=4, classifier__max_features=sqrt, clas

In [7]:
best_model = search.best_estimator_
predictions = best_model.predict(X_test)
metrics = {
    'model': MODEL_NAME,
    'features': FEATURES,
    'best_params': search.best_params_,
    'cv_macro_f1': float(search.best_score_),
    'test_accuracy': float(accuracy_score(y_test, predictions)),
    'test_balanced_accuracy': float(balanced_accuracy_score(y_test, predictions)),
    'test_macro_f1': float(f1_score(y_test, predictions, average='macro', zero_division=0)),
    'test_fatal_recall': float(recall_score(y_test, predictions, labels=[1], average=None, zero_division=0)[0]),
    'test_serious_recall': float(recall_score(y_test, predictions, labels=[2], average=None, zero_division=0)[0]),
    'test_slight_recall': float(recall_score(y_test, predictions, labels=[3], average=None, zero_division=0)[0]),
    'test_confusion_matrix': confusion_matrix(y_test, predictions, labels=[1, 2, 3]).tolist(),
}
(RESULTS / 'best_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
pd.DataFrame({'collision_index': test[GROUP], 'actual': y_test, 'predicted': predictions}).to_csv(RESULTS / 'test_predictions.csv', index=False)
joblib.dump(best_model, BEST / 'decision_tree_best.pkl')
joblib.dump(best_model, MODEL_DIR / 'decision_tree_2.pkl')
(BEST / 'decision_tree_best_params.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))

{
  "model": "decision_tree",
  "features": [
    "casualty_class",
    "sex_of_casualty",
    "age_of_casualty",
    "casualty_type",
    "number_of_vehicles",
    "day_of_week",
    "first_road_class",
    "road_type",
    "speed_limit",
    "junction_detail",
    "junction_control",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "urban_or_rural_area",
    "vehicle_type",
    "vehicle_manoeuvre",
    "vehicle_direction_from",
    "vehicle_direction_to",
    "vehicle_leaving_carriageway",
    "first_point_of_impact",
    "hit_object_off_carriageway",
    "age_of_driver",
    "engine_capacity_cc",
    "journey_purpose_of_driver"
  ],
  "best_params": {
    "classifier__min_samples_split": 50,
    "classifier__min_samples_leaf": 5,
    "classifier__max_features": 0.7,
    "classifier__max_depth": 24,
    "classifier__criterion": "gini",
    "classifier__class_weight": {
      "1": 3,
      "2": 1.5,
      "3": 1
    },
    "classifier__ccp_alpha": 0